In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import median_absolute_error
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import cross_val_predict
from scipy import stats

In [2]:
mlb_pitchers = pd.read_csv('Pitchers Cleaned.csv')
mlb_pitchers

,Player,Debut Year,Debut Age,Retirement Year,Retirement Age,Career Length,Wins,Losses,Win Percentage,Total Decisions,...,BF,ERA+,FIP,WHIP,H9,HR9,BB9,SO9,SO/BB,Hall of Fame
0,Cy Young,1890,23,1911,44,21,511,315,0.619,826,...,29565,138,2.84,1.130,8.7,0.2,1.5,3.4,2.30,1
1,Pud Galvin,1875,18,1892,35,17,365,310,0.541,675,...,25415,107,2.96,1.191,9.6,0.2,1.1,2.7,2.43,1
2,Walter Johnson,1907,19,1927,39,20,417,279,0.599,696,...,23415,147,2.38,1.061,7.5,0.1,2.1,5.3,2.57,1
3,Phil Niekro,1964,25,1987,48,23,318,274,0.537,592,...,22677,115,3.62,1.268,8.4,0.8,3.0,5.6,1.85,1
4,Nolan Ryan,1966,19,1993,46,27,324,292,0.526,616,...,22575,112,2.97,1.247,6.6,0.5,4.7,9.5,2.04,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1210,Charlie Robertson,1919,23,1928,32,9,49,80,0.380,129,...,4453,90,3.89,1.518,10.3,0.3,3.4,2.8,0.82,0
1211,Sammy Ellis,1962,21,1969,28,7,63,58,0.521,121,...,4296,88,3.90,1.340,8.7,1.1,3.4,6.1,1.79,0
1212,Lil Stoner,1922,23,1931,32,9,50,57,0.467,107,...,4466,87,4.13,1.548,10.6,0.6,3.4,2.7,0.80,0
1213,Johnny Humphries,1938,23,1946,31,8,52,63,0.452,115,...,4342,97,3.80,1.394,9.2,0.4,3.4,2.8,0.85,0


# RFC with KFold

In [3]:
rfc = RandomForestClassifier()

# Features
X = mlb_pitchers.drop(columns=['Hall of Fame', 'Player'])

# Target
y = mlb_pitchers['Hall of Fame']

# Split into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rfc = RandomForestClassifier()
rfc.fit(X_train, y_train)

RandomForestClassifier()

In [4]:
from sklearn.model_selection import cross_val_predict

# Generate out-of-fold predictions for all data
kf = KFold(n_splits=5, shuffle=True, random_state=42)
y_pred_all = cross_val_predict(rfc, X, y, cv=kf)


# Get predicted probabilities
y_proba_all = cross_val_predict(rfc, X, y, cv=kf, method='predict_proba')

# Regular predictions
y_pred_all = cross_val_predict(rfc, X, y, cv=kf)


# If binary classification, pick the probability of the predicted class
# (confidence in whichever class was chosen)
y_pred_indices = np.array([list(np.unique(y)).index(p) for p in y_pred_all])
confidence_scores = y_proba_all[np.arange(len(y_proba_all)), y_pred_indices]

# Build results dataframe
rfc_cv_results = pd.DataFrame()
rfc_cv_results['Player'] = mlb_pitchers['Player']
rfc_cv_results['Hall of Fame'] = y
rfc_cv_results['Prediction'] = y_pred_all
rfc_cv_results['Prediction Correct'] = (rfc_cv_results['Hall of Fame'] == rfc_cv_results['Prediction']).astype(int)
rfc_cv_results['Prediction Confidence'] = confidence_scores

#df['percentage_of_max'] = (df['values'] / max_value) * 100


rfc_cv_results

,Player,Hall of Fame,Prediction,Prediction Correct,Prediction Confidence
0,Cy Young,1,1,1,0.86
1,Pud Galvin,1,1,1,0.68
2,Walter Johnson,1,1,1,0.88
3,Phil Niekro,1,1,1,0.77
4,Nolan Ryan,1,1,1,0.82
...,...,...,...,...,...
1210,Charlie Robertson,0,0,1,1.00
1211,Sammy Ellis,0,0,1,1.00
1212,Lil Stoner,0,0,1,1.00
1213,Johnny Humphries,0,0,1,1.00


In [5]:
rfc_cv_results.to_csv('Results RFC no GS.csv', index=False)

In [ ]:
rfc_cv_results['RFC Confidence Score'].hist(bins=20)
plt.title('Confidence vs Reality Distribution')
plt.xlabel('Confidence Score (+ = correct, - = wrong)')
plt.ylabel('Count')
plt.show()

# Confidence Score

In [ ]:
# Score = +confidence if correct, -confidence if wrong
rfc_cv_results['RFC Confidence Score'] = np.where(
    rfc_cv_results['RFC Correct'] == 1,
    rfc_cv_results['RFC Confidence'],
    -rfc_cv_results['RFC Confidence']
)

# Make each result positive
rfc_cv_results['RFC Confidence Score'] = rfc_cv_results['RFC Confidence Score'] + 1

max_value = rfc_cv_results['RFC Confidence Score'].max()
rfc_cv_results['Confidence as Pct of Max'] = (rfc_cv_results['RFC Confidence Score'] / max_value) * 100

#df['percentage_of_max'] = (df['values'] / max_value) * 100


rfc_cv_results

In [24]:
import numpy as np

# signed score (you already have)
rfc_cv_results['RFC Signed Score'] = np.where(
    rfc_cv_results['RFC Correct'] == 1,
    rfc_cv_results['RFC Confidence'],
    -rfc_cv_results['RFC Confidence']
)

# Option 2: rescale signed score to [0,1]
rfc_cv_results['RFC Rescaled [0,1]'] = (1 + rfc_cv_results['RFC Signed Score']) / 2

# Option 3: reward positive, punishment mapped to (1 - p) so everything is in [0,1]
rfc_cv_results['RFC RewardPos'] = np.where(
    rfc_cv_results['RFC Correct'] == 1,
    rfc_cv_results['RFC Confidence'],
    1 - rfc_cv_results['RFC Confidence']
)

# Option 4: absolute confidence magnitude (loses correctness info)
rfc_cv_results['RFC AbsConfidence'] = rfc_cv_results['RFC Confidence'].abs()

# Example derived summaries
mean_signed = rfc_cv_results['RFC Signed Score'].mean()
mean_rescaled = rfc_cv_results['RFC Rescaled [0,1]'].mean()
mean_rewardpos = rfc_cv_results['RFC RewardPos'].mean()

print(f"Mean signed score ([-1,1]): {mean_signed:.4f}")
print(f"Mean rescaled score ([0,1]): {mean_rescaled:.4f}")
print(f"Mean reward-pos ([0,1]): {mean_rewardpos:.4f}")

Mean signed score ([-1,1]): 0.9063
Mean rescaled score ([0,1]): 0.9532
Mean reward-pos ([0,1]): 0.9417


In [25]:
rfc_cv_results

,Player,Hall of Fame,RFC Prediction,RFC Correct,RFC Confidence,RFC Signed Score,"RFC Rescaled [0,1]",RFC RewardPos,RFC AbsConfidence
0,Cy Young,1,1,1,0.91,0.91,0.955,0.91,0.91
1,Pud Galvin,1,1,1,0.59,0.59,0.795,0.59,0.59
2,Walter Johnson,1,1,1,0.89,0.89,0.945,0.89,0.89
3,Phil Niekro,1,1,1,0.77,0.77,0.885,0.77,0.77
4,Nolan Ryan,1,1,1,0.86,0.86,0.930,0.86,0.86
...,...,...,...,...,...,...,...,...,...
1210,Charlie Robertson,0,0,1,1.00,1.00,1.000,1.00,1.00
1211,Sammy Ellis,0,0,1,1.00,1.00,1.000,1.00,1.00
1212,Lil Stoner,0,0,1,1.00,1.00,1.000,1.00,1.00
1213,Johnny Humphries,0,0,1,1.00,1.00,1.000,1.00,1.00


In [28]:
rfc_cv_results.nsmallest(10, 'RFC Rescaled [0,1]')

,Player,Hall of Fame,RFC Prediction,RFC Correct,RFC Confidence,RFC Signed Score,"RFC Rescaled [0,1]",RFC RewardPos,RFC AbsConfidence
15,Roger Clemens,0,1,0,0.97,-0.97,0.015,0.03,0.97
91,Rube Marquard,1,0,0,0.97,-0.97,0.015,0.03,0.97
19,Tommy John,0,1,0,0.96,-0.96,0.020,0.04,0.96
105,Jesse Haines,1,0,0,0.96,-0.96,0.020,0.04,0.96
152,Jack Chesbro,1,0,0,0.92,-0.92,0.040,0.08,0.92
653,Andy Cooper,1,0,0,0.86,-0.86,0.070,0.14,0.86
246,Lefty Gomez,1,0,0,0.85,-0.85,0.075,0.15,0.85
257,John Ward,1,0,0,0.85,-0.85,0.075,0.15,0.85
23,Tony Mullane,0,1,0,0.83,-0.83,0.085,0.17,0.83
1174,Bruce Sutter,1,0,0,0.83,-0.83,0.085,0.17,0.83


# Histogram

In [ ]:
rfc_cv_results['RFC Confidence Score'].hist(bins=20)
plt.title('Confidence vs Reality Distribution')
plt.xlabel('Confidence Score (+ = correct, - = wrong)')
plt.ylabel('Count')
plt.show()